In [ ]:
# Imports & paths
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_RAW = Path("../data/raw")
files = [
    "decimal_benign.csv",
    "decimal_DoS.csv",
    "decimal_spoofing-GAS.csv",
    "decimal_spoofing-RPM.csv",
    "decimal_spoofing-SPEED.csv",
    "decimal_spoofing-STEERING_WHEEL.csv"
]

print(pd.read_csv('../data/raw/data_description.csv'), sep=',')

In [ ]:
dfs_clean = []

for file_name in files:
    path = DATA_RAW / file_name
    
    # Read
    df_temp = pd.read_csv(path, dtype=str) 
    # Strip whitespace from ALL column names
    df_temp.columns = df_temp.columns.str.strip()
    
    # Drop any duplicate columns that might still exist after rename
    # (keeps the first occurrence)
    df_temp = df_temp.loc[:, ~df_temp.columns.duplicated()]
    
    # Step 4: Add source tracking
    df_temp['source_file'] = file_name
    
    # Optional: Ensure core columns exist
    expected = ['ID', 'DATA_0', 'DATA_1', 'DATA_2', 'DATA_3',
                'DATA_4', 'DATA_5', 'DATA_6', 'DATA_7', 'label', 'category', 'specific_class', 'source_file']
    missing = [col for col in expected if col not in df_temp.columns]
    if missing:
        print(f"Warning: {file_name} missing columns: {missing}")
    else:
        print("No missing columns.")
    
    dfs_clean.append(df_temp)

# Concatenate
df = pd.concat(dfs_clean, ignore_index=True)


# checks

print("\nFinal shape:", df.shape)
print("Final columns:", df.columns.tolist())
print("\nUnique labels:", sorted(df['label'].unique()))
print("\nUnique categories:", df['category'].dropna().unique() if 'category' in df else "No category column")
print("\nUnique specific_classes:", df['specific_class'].dropna().unique() if 'specific_class' in df else "No specific_class column")

In [ ]:
# Final label/category/specific_class cleaning and normalization
df['label'] = df['label'].str.strip().str.upper()
df['category'] = df['category'].str.strip().str.upper()
df['specific_class'] = df['specific_class'].str.strip().str.upper()
# Optional: Create binary target for quick baseline (if you want binary first)
df['is_attack'] = (df['label'] == 'ATTACK').astype(int)

# FInal checks
print("Final shape:", df.shape)
print("Final columns:", df.columns.tolist())
print("\nUnique labels:", sorted(df['label'].unique()))
print("\nUnique categories:", df['category'].dropna().unique() if 'category' in df else "No category column")
print("\nUnique specific_classes:", df['specific_class'].dropna().unique() if 'specific_class' in df else "No specific_class column")

In [ ]:
# Quick visuals
plt.figure(figsize=(10,6))
sns.countplot(y='label', data=df, order=df['label'].value_counts().index)
plt.title("Class Distribution")
plt.show()

In [ ]:
df.head()

In [ ]:
# Convert numerics
num_cols = ['ID', 'DATA_0', 'DATA_1', 'DATA_2', 'DATA_3', 'DATA_4', 'DATA_5', 'DATA_6', 'DATA_7']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce', downcast='integer')

#Duplicates check
print("Final shape:", df.shape)
print("Full duplicates:", df.duplicated().sum())                    # Likely very high
print("Duplicates ignoring label/category:", df.drop(['label','category','specific_class','source_file'], axis=1).duplicated().sum())

In [ ]:
# =============================================================================
# Per-Class Deduplication → Preserve Unique Signatures per Class
# =============================================================================

# Define what constitutes a "signature"
signature_cols = ['ID', 'DATA_0', 'DATA_1', 'DATA_2', 'DATA_3',
                  'DATA_4', 'DATA_5', 'DATA_6', 'DATA_7']

# We'll use 'specific_class' as the grouping key (most granular)
group_key = 'specific_class'

# List of classes 
classes = ['BENIGN', 'DOS', 'GAS', 'RPM', 'SPEED', 'STEERING_WHEEL']

dfs_deduped = []

for cls in classes:
    # Subset for this class
    df_cls = df[df[group_key] == cls].copy()
    
    orig_rows = len(df_cls)
    
    # Deduplicate within this class only
    df_cls_dedup = df_cls.drop_duplicates(subset=signature_cols)
    
    dedup_rows = len(df_cls_dedup)
    removed = orig_rows - dedup_rows
    
    print(f"{cls:16} | Original: {orig_rows:>8,} | After dedup: {dedup_rows:>6,} | Removed: {removed:>6,}")
    
    dfs_deduped.append(df_cls_dedup)

# Concatenate all deduplicated subsets
df_dedup_per_class = pd.concat(dfs_deduped, ignore_index=True)

print("\n" + "="*60)
print(f"Final deduplicated shape: {df_dedup_per_class.shape}")
print(f"Total unique signatures across classes: {len(df_dedup_per_class):,}")

# Distribution after per-class dedup
print("\nFinal class distribution:")
print(df_dedup_per_class[group_key].value_counts().to_frame(name='count'))

In [ ]:
# =============================================================================
# Cell: Meta-Feature Generation via Out-of-Fold Predictions
# =============================================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder

# Use the deduplicated dataset (per-class version)
df_base = df_dedup_per_class.copy()

In [ ]:
# Features & target
X = df_base[signature_cols]
target_col = 'label'
le = LabelEncoder()
y = le.fit_transform(df_base[target_col])

In [ ]:
# Prepare arrays for OOF predictions
n_classes = len(le.classes_)
oof_probs = np.zeros((len(X), n_classes), dtype=float)
oof_predictions = np.zeros(len(X), dtype=int)   # optional hard predictions

In [ ]:
oof_probs

In [ ]:
oof_predictions

In [ ]:






# Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"Fold {fold+1}/5 ...")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Train simple base model
    meta_model = RandomForestClassifier(
        n_estimators=80,
        max_depth=7,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    )
    meta_model.fit(X_train, y_train)
    
    # Predict probabilities on validation fold
    probs = meta_model.predict_proba(X_val)
    oof_probs[val_idx] = probs
    
    # Optional: hard prediction
    oof_predictions[val_idx] = meta_model.predict(X_val)
    
    # Optional quality check
    fold_logloss = log_loss(y_val, probs)
    print(f"  Fold log-loss: {fold_logloss:.4f}")

# Add meta-features to dataframe
for i, cls_name in enumerate(le.classes_):
    df_base[f'prob_{cls_name}'] = oof_probs[:, i]

# Optional: add hard prediction as categorical feature
df_base['meta_pred'] = le.inverse_transform(oof_predictions)

print("\nNew columns added:")
print([c for c in df_base.columns if c.startswith('prob_') or c == 'meta_pred'])

# Quick correlation check (optional)
print("\nCorrelation between meta-probs and is_attack:")
print(df_base[[f'prob_{cls}' for cls in le.classes_ if 'ATTACK' in cls or 'DOS' in cls or 'SPOOFING' in cls] + ['is_attack']].corr()['is_attack'])

In [ ]:
df_base

In [ ]:
# =============================================================================
# Cell: ANOVA + RF Baseline on Deduplicated Data
# =============================================================================

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Use deduplicated df
df_model = df_dedup_per_class.copy()

signature_cols = ['ID', 'DATA_0', 'DATA_1', 'DATA_2', 'DATA_3', 'DATA_4', 'DATA_5', 'DATA_6', 'DATA_7']
target_col = 'specific_class'  # 6-class - hardest, closest to your paper

X = df_model[signature_cols]
y_str = df_model[target_col]
le = LabelEncoder()
y = le.fit_transform(y_str)

# 1. ANOVA feature selection
selector = SelectKBest(f_classif, k='all')
selector.fit(X, y)

feat_importance = pd.DataFrame({
    'Feature': signature_cols,
    'F-score': selector.scores_,
    'p-value': selector.pvalues_
}).sort_values('F-score', ascending=False)

print("ANOVA Feature Ranking:\n", feat_importance)

# Select top features (e.g., p < 0.05 or top 6-7)
selected_features = feat_importance[feat_importance['p-value'] < 0.05]['Feature'].tolist()
if len(selected_features) < 4:
    selected_features = feat_importance.head(6)['Feature'].tolist()  # fallback

print("\nSelected features:", selected_features)
X_sel = X[selected_features]

# 2. Stratified split (but with tiny minorities, use CV instead)
# For tiny classes, better: StratifiedKFold with adjusted folds

skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
macro_f1_scores = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_sel, y)):
    X_tr, X_te = X_sel.iloc[train_idx], X_sel.iloc[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]
    
    rf = RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced_subsample',
        max_depth=7,               # limit to avoid overfit on tiny data
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_tr, y_tr)
    
    y_pred = rf.predict(X_te)
    f1 = f1_score(y_te, y_pred, average='macro', zero_division=0)
    macro_f1_scores.append(f1)
    print(f"Fold {fold+1} macro-F1: {f1:.4f}")

print("\nCV Macro-F1 mean ± std: {:.4f} ± {:.4f}".format(
    np.mean(macro_f1_scores), np.std(macro_f1_scores)))